In [1]:
!pip install faker pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.4 MB/s eta 0:00:00


Adding randomness and creating template for datasets

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random

fake = Faker()

random.seed(42)
np.random.seed(42)
Faker.seed(42)

CITIES = ["Bangalore", "Mumbai", "Delhi", "Chennai", "Hyderabad", "Pune", "Kolkata"]

def generate_clean_data(n_rows=50):
    data = []

    for _ in range(n_rows):
        row = {
            "name": fake.name(),
            "age": random.randint(18, 70),
            "email": fake.email(),
            "city": random.choice(CITIES),
            "salary": round(random.uniform(20000, 100000), 2)
        }
        data.append(row)

    df = pd.DataFrame(data)

    return df

In [3]:
df_clean = generate_clean_data(20)

df_clean.head()

,name,age,email,city,salary
0,Allison Hill,58,donaldgarcia@example.net,Bangalore,22000.86
1,Angie Henderson,35,davisjesse@example.net,Mumbai,37856.86
2,Cristian Santos,65,lrobinson@example.com,Bangalore,74135.96
3,Abigail Shaffer,52,jpeterson@example.org,Bangalore,67239.40
4,Gabrielle Davis,20,howardmaurice@example.com,Bangalore,27495.62


Checking dataset before corruption

In [4]:
print("Missing values:\n", df_clean.isnull().sum())
print("\nDuplicates:", df_clean.duplicated().sum())
print("\nData types:\n", df_clean.dtypes)

Missing values:
 name      0
age       0
email     0
city      0
salary    0
dtype: int64

Duplicates: 0

Data types:
 name       object
age         int64
email      object
city       object
salary    float64
dtype: object


Corrupting the dataset

In [5]:
def inject_missing_values(df, fraction=0.1):
    df = df.copy()

    n_rows, n_cols = df.shape
    total_cells = n_rows * n_cols
    n_missing = int(total_cells * fraction)

    for _ in range(n_missing):
        i = random.randint(0, n_rows - 1)
        j = random.randint(0, n_cols - 1)
        df.iat[i, j] = np.nan

    return df

In [6]:
def inject_duplicates(df, fraction=0.1):
    df = df.copy()

    n_rows = df.shape[0]
    n_duplicates = int(n_rows * fraction)

    duplicate_rows = df.sample(n=n_duplicates, replace=True)

    df = pd.concat([df, duplicate_rows], ignore_index=True)

    return df

In [7]:
def corrupt_salary_format(df, fraction=0.3):
    df = df.copy()

    # Convert entire column to object first (IMPORTANT)
    df["salary"] = df["salary"].astype(object)

    indices = df.sample(frac=fraction).index

    for i in indices:
        val = df.at[i, "salary"]
        df.at[i, "salary"] = f"${val:,.2f}"

    return df

In [8]:
def corrupt_dataset(df):
    df_dirty = df.copy()

    df_dirty = inject_missing_values(df_dirty, 0.1)
    df_dirty = inject_duplicates(df_dirty, 0.1)
    df_dirty = corrupt_salary_format(df_dirty, 0.3)

    return df_dirty

Checking the dataset after corruption

In [9]:
df_dirty = corrupt_dataset(df_clean)

df_dirty.head(10)

,name,age,email,city,salary
0,Allison Hill,58.0,donaldgarcia@example.net,Bangalore,"$22,000.86"
1,Angie Henderson,35.0,davisjesse@example.net,Mumbai,37856.86
2,Cristian Santos,65.0,lrobinson@example.com,Bangalore,74135.96
3,Abigail Shaffer,52.0,jpeterson@example.org,Bangalore,67239.4
4,Gabrielle Davis,20.0,howardmaurice@example.com,Bangalore,27495.62
5,Monica Herrera,32.0,NaN,NaN,68161.5
6,Shannon Ray,53.0,williamsjeremy@example.com,Mumbai,77281.57
7,NaN,62.0,xreid@example.org,Hyderabad,53561.59
8,NaN,46.0,lynchgeorge@example.net,Hyderabad,"$42,255.26"
9,NaN,18.0,gabriellecameron@example.org,Kolkata,"$84,465.54"


In [10]:
print("Missing values:\n", df_dirty.isnull().sum())
print("\nDuplicates:", df_dirty.duplicated().sum())
print("\nData types:\n", df_dirty.dtypes)

Missing values:
 name      3
age       4
email     3
city      1
salary    0
dtype: int64

Duplicates: 2

Data types:
 name       object
age       float64
email      object
city       object
salary     object
dtype: object


Function that will be called during reset

In [11]:
def generate_episode(n_rows=50):
    # Step 1: generate clean dataset
    df_clean = generate_clean_data(n_rows)

    # Step 2: corrupt it
    df_dirty = corrupt_dataset(df_clean)

    return df_clean, df_dirty

In [12]:
for i in range(3):
    clean, dirty = generate_episode(20)

    print(f"\n--- Episode {i+1} ---")
    print("Clean shape:", clean.shape)
    print("Dirty shape:", dirty.shape)
    print("Duplicates:", dirty.duplicated().sum())


--- Episode 1 ---
Clean shape: (20, 5)
Dirty shape: (22, 5)
Duplicates: 1

--- Episode 2 ---
Clean shape: (20, 5)
Dirty shape: (22, 5)
Duplicates: 0

--- Episode 3 ---
Clean shape: (20, 5)
Dirty shape: (22, 5)
Duplicates: 1


In [13]:
def run_pipeline(num_runs=5):
    for i in range(num_runs):
        clean, dirty = generate_episode(20)

        print(f"\n=== Run {i+1} ===")
        print("Dirty dataset preview:")
        print(dirty.head(3))
run_pipeline(3)


=== Run 1 ===
Dirty dataset preview:
                name   age                        email       city      salary
0                NaN  33.0   elliottjeffery@example.net  Hyderabad    67561.53
1                NaN  57.0  stricklandfrank@example.com  Bangalore  $53,537.99
2  Alexander Collins  55.0         tsanders@example.org  Hyderabad    61822.62

=== Run 2 ===
Dirty dataset preview:
             name  age                      email       city    salary
0  Jennifer Jones   22  hernandezlisa@example.com       Pune  92314.29
1     David Grant   52        yobrien@example.net  Bangalore   86767.6
2    Joseph Hayes   55                        NaN  Hyderabad   31847.5

=== Run 3 ===
Dirty dataset preview:
            name   age                        email       city      salary
0  Thomas Romero  45.0     michellehill@example.com      Delhi     78283.6
1  Richard Adams   NaN  campbellkenneth@example.net    Chennai  $68,496.71
2    Mary Grimes  50.0                          NaN  Bangalor

testing

In [14]:
generate_episode()

(                  name  age                        email       city    salary
 0      Patricia Becker   67   courtneyberger@example.net  Bangalore  56433.29
 1      Richard Johnson   58        jessica14@example.com  Hyderabad  35554.82
 2       Casey Anderson   62         alyssa42@example.com    Chennai  59550.17
 3         Terry Coffey   33          julie51@example.com     Mumbai  72484.64
 4          Brenda Levy   18          zcoffey@example.net    Kolkata  88835.25
 5        Richard Smith   24      gibsonemily@example.net    Kolkata  54011.70
 6         James Little   29         daniel37@example.org    Kolkata  96637.28
 7         Terri Murphy   51      deborahreid@example.com    Chennai  24017.47
 8      Elizabeth Ortiz   33   brandonjohnson@example.com    Kolkata  29708.69
 9         David Medina   26          qchavez@example.net    Kolkata  57171.46
 10   Kimberly Matthews   51       nicolepena@example.com  Hyderabad  67636.19
 11      Teresa Ramirez   66           jeff73@exampl

In [15]:
def analyze_issues(df):
    issues = {}

    issues["missing"] = df.isnull().sum().sum()
    issues["duplicates"] = df.duplicated().sum()
    issues["salary_format"] = df["salary"].apply(lambda x: isinstance(x, str)).sum()

    return issues

reward systen

In [16]:
def compute_reward(before_df, after_df):

    before = analyze_issues(before_df)
    after = analyze_issues(after_df)

    reward = 0

    # Missing values improvement
    reward += (before["missing"] - after["missing"]) * 0.1

    # Duplicate improvement
    reward += (before["duplicates"] - after["duplicates"]) * 0.2

    # Salary format improvement
    reward += (before["salary_format"] - after["salary_format"]) * 0.1

    # Penalty if things got worse
    if reward < 0:
        reward -= 0.2

    return reward

In [17]:
df_before = df_dirty.copy()

df_after = apply_action(df_before, "remove_duplicates")

reward = compute_reward(df_before, df_after)

print("Reward:", reward)

NameError: name 'apply_action' is not defined

In [ ]:
df_before = df_dirty.copy()

df_step1 = apply_action(df_before, "remove_duplicates")
print("Reward step 1:", compute_reward(df_before, df_step1))

df_step2 = apply_action(df_step1, "fill_missing_values")
print("Reward step 2:", compute_reward(df_step1, df_step2))

df_step3 = apply_action(df_step2, "fix_salary_format")
print("Reward step 3:", compute_reward(df_step2, df_step3))